# 10 — Host Holdout: Realistic Class-1 (Mode-A) Benchmark

Actually SCORES `enveda-np-examples` (the locally available primary host-like holdout notebook
05 detected but never scored) as the project's realistic Class-1 benchmark -- quantifying how
optimistic notebook 09's `0.7531` known-spectrum MRR@25 is, since that number comes from a
RANDOM 996-query dev sample that shares the SAME `ingest_lib` distribution as its own reference
library. `enveda-np-examples` is timsTOF-only and was NEVER used to train or select the
LambdaMART model scored here, so it stands in for what an unpublished, host-generated test
spectrum can realistically expect.

Scope this notebook actually covers (spec Parts 4, 5, 6, 7 -- Parts 8-19, the test-like
multiplicity resample, calibration, mass log-likelihood, molecule aggregation, first submission,
and offline smoke test, are explicitly NOT covered here; see the closing section for why and
what's next):

```text
4. host holdout artifacts + strict training exclusion
5. near-duplicate reference audit (casmi.spectra.deduplication)
6. host Mode-A baselines: mass-only, peak-overlap, LambdaMART -- standard vs. cross-library vs.
   duplicate-aware protocols, reported separately
7. paired bootstrap CIs for every comparison
```

Every number below is computed live against the real `enveda-np-examples` spectra (1,184
spectra / 250 connectivities, loaded programmatically -- never hard-coded).

In [1]:
import sys, json, time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import lightgbm as lgb

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact, artifact_fingerprint, save_artifact, artifact_is_valid, file_fingerprint
from casmi.io.parquet import load_rows_by_offset
from casmi.chemistry.adducts import neutral_mass_from_precursor
from casmi.candidates.cache import cached_mass_index, cached_dataframe, upstream_fingerprints
from casmi.candidates.mass_index import MassIndex
from casmi.candidates.generator import CandidateGenerator, dedupe_pool_to_connectivity
from casmi.spectra.preprocessing import remove_invalid_peaks, truncate_top_peaks
from casmi.spectra.library import build_reference_index
from casmi.spectra.deduplication import find_near_duplicate_references
from casmi.ranking.features import compute_pair_spectral_scores
from casmi.ranking.feature_policy import SPECTRAL_RANKING_FEATURE_POLICY, assert_no_prohibited_features
from casmi.ranking.evaluation import rank_candidates, ranking_metrics, true_candidate_rank, per_query_reciprocal_rank
from casmi.validation.checks import CheckResult, summarize_checks
from casmi.validation.baseline_registry import load_baseline, load_promoted_baseline, register_baseline
from casmi.validation.bootstrap import paired_bootstrap_report
from casmi.data.aggregation import build_molecule_mass_variants

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
N_BOOTSTRAP = 5000
HOST_INGEST_LIB = "enveda-np-examples"
MAX_REFS_PER_CANDIDATE = 5
BIN_WIDTH_DA = 0.1
PEAK_TOL_DA = 0.02
MAX_PEAKS_PER_SPECTRUM = 100
DEDUP_COSINE_THRESHOLD = 0.95
DEDUP_PRECURSOR_DIFF_DA = 0.01
LGBM_PARAMS = dict(
    objective="lambdarank", metric="ndcg", n_estimators=300, learning_rate=0.05,
    num_leaves=31, min_child_samples=20, random_state=SEED, verbosity=-1,
)

paths = get_project_paths()
CG_INTERIM = paths.interim / "candidate_generation"
CG_PROCESSED = paths.processed / "candidate_generation"
RANK_INTERIM = paths.interim / "spectral_ranking"
HOST_INTERIM = paths.interim / "host_holdout"
HOST_PROCESSED = paths.processed / "host_holdout"
for d in (HOST_INTERIM, HOST_PROCESSED):
    d.mkdir(parents=True, exist_ok=True)
BASELINE_REGISTRY_PATH = paths.processed / "baseline_registry.json"
REPORTS_DIR = paths.reports

MODEL_FEATURES = SPECTRAL_RANKING_FEATURE_POLICY.model_features
checks = []
print(f"MODEL_FEATURES: {MODEL_FEATURES}")

MODEL_FEATURES: ['abs_mass_error_ppm', 'cosine_max', 'cosine_top3_mean', 'modified_cosine_max', 'modified_cosine_top3_mean', 'peak_overlap_frac_max', 'peak_overlap_frac_top3_mean', 'neutral_loss_cosine_max', 'neutral_loss_cosine_top3_mean']


## 1. Load upstream artifacts (no recomputation of notebook 01-03's own outputs)

In [2]:
molecule_metadata = load_artifact("molecule_metadata")
train_meta = load_artifact("train_spectrum_metadata")
connectivity_folds = load_artifact("connectivity_folds")

with open(CG_PROCESSED / "candidate_generation_contract.json") as f:
    contract = json.load(f)
SELECTED_PPM = contract["molecule_policy"]["tolerance_ppm"]
print(f"candidate-generation contract status: {contract['status']}, operating tolerance: {SELECTED_PPM} ppm")

molecule_mass_variants = pd.read_parquet(CG_INTERIM / "molecule_mass_variants.parquet")
library_fp = artifact_fingerprint("molecule_mass_variants", CG_INTERIM)
mass_index, _, _ = cached_mass_index(
    CG_INTERIM, build_fn=lambda: MassIndex.from_dataframe(molecule_mass_variants, mass_col="exact_mass", key_col="mass_variant_id"),
    library_fingerprint=library_fp, mass_col="exact_mass", key_col="mass_variant_id",
)
variant_to_connectivity = molecule_mass_variants.set_index("mass_variant_id")["connectivity_key"]
gen = CandidateGenerator(mass_index)

connectivity_to_fold = connectivity_folds.set_index("connectivity_key")["fold"].to_dict()
spectrum_id_to_ingest_lib = train_meta.set_index("train_spectrum_id")["ingest_lib"].to_dict()
reference_index = build_reference_index(train_meta, connectivity_col="connectivity_key", id_col="train_spectrum_id")

dev_ranking_queries = pd.read_parquet(RANK_INTERIM / "dev_ranking_queries.parquet")
pair_features_dev_known = pd.read_parquet(RANK_INTERIM / "pair_features_dev_known_spectrum.parquet")
print(f"train_spectrum_metadata: {train_meta.shape}, connectivity_folds: {connectivity_folds.shape}")
checks.append(CheckResult("upstream artifacts loaded", True, "notebook 01/02/03/04 artifacts reloaded, no recomputation"))

candidate-generation contract status: PROVISIONAL, operating tolerance: 50 ppm
[CACHE HIT] mass_index <- C:\Users\myben\OneDrive\Documents\Enveda\data\interim\candidate_generation\mass_index.npz (library_fingerprint matches)


train_spectrum_metadata: (2539608, 27), connectivity_folds: (274195, 2)


## 2. Section 4.1 — Primary host-like holdout artifacts

`ingest_lib == "enveda-np-examples"` inside `train_spectrum_metadata` -- this holdout is NOT a separate downloaded dataset, it's a real slice of `train.parquet` notebook 05 already discovered. Every scale number below is `len(...)`/`.nunique()` on the live data, never hard-coded.

In [3]:
host_holdout_spectra = train_meta[train_meta["ingest_lib"] == HOST_INGEST_LIB].reset_index(drop=True).rename(
    columns={"train_spectrum_id": "query_id", "connectivity_key": "true_connectivity_key"}
)
host_holdout_spectra["neutral_mass"] = [
    neutral_mass_from_precursor(mz, a) for mz, a in zip(host_holdout_spectra["precursor_mz"], host_holdout_spectra["adduct"])
]
host_holdout_spectra["fold"] = host_holdout_spectra["true_connectivity_key"].map(connectivity_to_fold)
assert host_holdout_spectra["fold"].notna().all(), "every host connectivity must have a fold assignment from the same GroupKFold as the rest of train"

host_grouping_cols = host_holdout_spectra.assign(
    instrument_type_str=host_holdout_spectra["instrument_type"].astype(str),
    adduct_str=host_holdout_spectra["adduct"].astype(str),
)
host_holdout_molecules = host_grouping_cols.groupby("true_connectivity_key").agg(
    n_spectra=("query_id", "count"), fold=("fold", "first"),
    instrument_types=("instrument_type_str", lambda s: sorted(s.unique().tolist())),
    adducts=("adduct_str", lambda s: sorted(s.unique().tolist())),
).reset_index().rename(columns={"true_connectivity_key": "connectivity_key"})

host_holdout_connectivity_keys = host_holdout_molecules[["connectivity_key", "fold"]].copy()

for name, df in [("host_holdout_spectra", host_holdout_spectra), ("host_holdout_molecules", host_holdout_molecules),
                  ("host_holdout_connectivity_keys", host_holdout_connectivity_keys)]:
    save_artifact(df, name, HOST_PROCESSED, description=f"Primary host-like holdout ({HOST_INGEST_LIB}) -- {name}")

print(f"host_holdout_spectra: {len(host_holdout_spectra):,} spectra, {host_holdout_spectra['true_connectivity_key'].nunique():,} connectivities")
print(f"instrument types: {sorted(host_holdout_spectra['instrument_type'].unique())}")
print(f"neutral_mass supported: {host_holdout_spectra['neutral_mass'].notna().mean():.2%}")
host_holdout_spectra = host_holdout_spectra[host_holdout_spectra["neutral_mass"].notna()].reset_index(drop=True)
all_host_query_ids = host_holdout_spectra["query_id"].tolist()
checks.append(CheckResult("host holdout artifacts saved", True, f"{len(host_holdout_spectra)} spectra, {len(host_holdout_molecules)} molecules"))

host_holdout_spectra: 1,184 spectra, 250 connectivities
instrument types: ['timsTOF']
neutral_mass supported: 100.00%


## 3. Section 4.2 — Strict training exclusion

Computed, not asserted-by-assumption: how much does notebook 09's 996-query known-spectrum training sample overlap with the 250 host connectivities? Every host query below is scored by the fold model that held ITS OWN connectivity's fold out (reproduced exactly as in notebook 09/09b) -- so even the 23-ish overlapping connectivities noted in notebook 05 never leak into the model that scores them.

In [4]:
dev_training_connectivities = set(dev_ranking_queries["true_connectivity_key"])
host_connectivities = set(host_holdout_spectra["true_connectivity_key"])
overlap = dev_training_connectivities & host_connectivities
print(f"{len(overlap)} / {len(host_connectivities)} host connectivities also appear as a TRUE connectivity in the 996-query LambdaMART training sample")
print(f"(matches notebook 05's ~23/250 finding -- not zero, but small, and irrelevant here: every host query is scored")
print(f" by the fold-held-out model for ITS OWN connectivity's fold, never the model that saw it, regardless of this overlap)")

checks.append(CheckResult("host/training connectivity overlap computed (informational, not a failure)", True,
                           f"{len(overlap)}/{len(host_connectivities)} overlap"))

23 / 250 host connectivities also appear as a TRUE connectivity in the 996-query LambdaMART training sample
(matches notebook 05's ~23/250 finding -- not zero, but small, and irrelevant here: every host query is scored
 by the fold-held-out model for ITS OWN connectivity's fold, never the model that saw it, regardless of this overlap)


## 4. Reproduce notebook 09's exact fold models (audit subject, not a new model)

In [5]:
train_table = pair_features_dev_known.merge(dev_ranking_queries[["query_id", "fold", "true_connectivity_key"]], on="query_id", how="left")
assert_no_prohibited_features(train_table[MODEL_FEATURES], SPECTRAL_RANKING_FEATURE_POLICY)

fold_models = {}
for held_out_fold in sorted(train_table["fold"].unique()):
    tr = train_table[train_table["fold"] != held_out_fold].sort_values("query_id")
    tr_group = tr.groupby("query_id", sort=True).size().to_numpy()
    model = lgb.LGBMRanker(**LGBM_PARAMS)
    model.fit(tr[MODEL_FEATURES], tr["is_true_candidate"].astype(int), group=tr_group)
    fold_models[held_out_fold] = model

mode_a_mrr25_registered = load_promoted_baseline("notebook_09_lambdamart", BASELINE_REGISTRY_PATH, regime="mode_a")["mrr_at_25"]
print(f"reproduced {len(fold_models)} fold models (registered notebook 09 Mode A MRR@25 for reference: {mode_a_mrr25_registered:.4f})")
checks.append(CheckResult("fold models reproduced", len(fold_models) == 5, f"{len(fold_models)} models"))

reproduced 5 fold models (registered notebook 09 Mode A MRR@25 for reference: 0.7531)


## 5. Section 5-6 candidates — generate host candidate pool (same mass index, same operating tolerance)

In [6]:
host_pool_raw = gen.generate_dataframe(
    host_holdout_spectra, tolerance_ppm=SELECTED_PPM,
    query_id_col="query_id", mass_col="neutral_mass", true_key_col="true_connectivity_key",
)
host_pool = dedupe_pool_to_connectivity(host_pool_raw, variant_to_connectivity)
print(f"host_pool: {len(host_pool):,} (query, candidate) pairs over {host_pool['query_id'].nunique():,} / {len(host_holdout_spectra):,} host queries")

host_true_present = set(host_pool.loc[host_pool["is_true_candidate"], "query_id"])
host_candidate_availability = len(host_true_present) / len(all_host_query_ids)
print(f"host candidate availability (true connectivity present in pool @ {SELECTED_PPM}ppm): {host_candidate_availability:.2%}")
checks.append(CheckResult("host candidate pool generated", len(host_pool) > 0, f"{len(host_pool):,} pairs, availability={host_candidate_availability:.2%}"))

host_pool: 126,573 (query, candidate) pairs over 1,184 / 1,184 host queries
host candidate availability (true connectivity present in pool @ 50ppm): 100.00%


## 6. Section 5 — load raw peaks + near-duplicate reference audit

One combined `load_rows_by_offset` call (query peaks + every needed reference spectrum's peaks), same pattern notebook 04 uses. Then `casmi.spectra.deduplication` flags, for the TRUE candidate's own reference pool specifically, which references are near-identical technical replicates of the query itself -- flagged, not deleted (spec 5.1).

In [7]:
def offset_of(spectrum_id):
    return int(spectrum_id.split("_")[1])


cap = MAX_REFS_PER_CANDIDATE
host_candidate_keys = host_pool["candidate_connectivity_key"].unique()
host_ref_ids_needed = set()
for key in host_candidate_keys:
    host_ref_ids_needed.update(reference_index.get(key, [])[: cap * 5])

host_query_offsets = set(host_holdout_spectra["query_id"].apply(offset_of))
host_ref_offsets = set(offset_of(i) for i in host_ref_ids_needed)
host_all_offsets = host_query_offsets | host_ref_offsets
print(f"host: {len(host_query_offsets):,} query offsets + {len(host_ref_offsets):,} reference offsets = {len(host_all_offsets):,} total")

t0 = time.time()
raw = load_rows_by_offset(paths.train, ["ms2_mzs", "ms2_normalized_intensities", "precursor_mz"], list(host_all_offsets))
raw = raw.rename(columns={"_row_offset": "row_offset"})
host_peaks_by_offset = {}
for row in raw.itertuples(index=False):
    mzs, ints = remove_invalid_peaks(row.ms2_mzs, row.ms2_normalized_intensities)
    mzs, ints = truncate_top_peaks(mzs, ints, max_peaks=MAX_PEAKS_PER_SPECTRUM)
    host_peaks_by_offset[row.row_offset] = {"mzs": mzs, "intensities": ints, "precursor_mz": row.precursor_mz}
print(f"loaded {len(host_peaks_by_offset):,} host-side peak rows in {time.time() - t0:.1f}s")
checks.append(CheckResult("host peak arrays loaded", len(host_peaks_by_offset) > 0, f"{len(host_peaks_by_offset):,} spectra"))

host: 1,184 query offsets + 197,403 reference offsets = 198,528 total


loaded 198,528 host-side peak rows in 35.1s


In [8]:
near_dup_rows = []
for query_row in host_holdout_spectra.itertuples(index=False):
    q_offset = offset_of(query_row.query_id)
    if q_offset not in host_peaks_by_offset:
        continue
    q_peaks = host_peaks_by_offset[q_offset]
    own_ref_ids = [rid for rid in reference_index.get(query_row.true_connectivity_key, []) if rid != query_row.query_id]
    ref_id_to_peaks = {rid: host_peaks_by_offset[offset_of(rid)] for rid in own_ref_ids if offset_of(rid) in host_peaks_by_offset}
    flagged = find_near_duplicate_references(q_peaks, ref_id_to_peaks, cosine_threshold=DEDUP_COSINE_THRESHOLD, precursor_diff_da=DEDUP_PRECURSOR_DIFF_DA)
    for rid in ref_id_to_peaks:
        near_dup_rows.append({
            "query_id": query_row.query_id, "reference_spectrum_id": rid,
            "reference_ingest_lib": spectrum_id_to_ingest_lib.get(rid),
            "is_near_duplicate_reference": rid in flagged,
        })
near_duplicate_audit = pd.DataFrame(near_dup_rows)
n_flagged = int(near_duplicate_audit["is_near_duplicate_reference"].sum()) if len(near_duplicate_audit) else 0
print(f"near-duplicate audit: {len(near_duplicate_audit):,} (query, own-connectivity-reference) pairs checked, {n_flagged:,} flagged as near-duplicates "
      f"(cosine>{DEDUP_COSINE_THRESHOLD}, precursor diff<{DEDUP_PRECURSOR_DIFF_DA}Da)")
if len(near_duplicate_audit):
    save_artifact(near_duplicate_audit, "host_near_duplicate_reference_audit", HOST_PROCESSED,
                   description="Section 5: which same-connectivity reference spectra are near-duplicate technical replicates of each host query")
checks.append(CheckResult("near-duplicate reference audit computed", True, f"{n_flagged}/{len(near_duplicate_audit)} flagged"))

near-duplicate audit: 35,773 (query, own-connectivity-reference) pairs checked, 1,332 flagged as near-duplicates (cosine>0.95, precursor diff<0.01Da)


## 7. Section 6.2 — build pair features under three reference protocols

**standard**: same-source references allowed, only the exact query spectrum excluded (what notebook 04's known-spectrum regime does). **cross_library**: also excludes every `enveda-np-examples` reference (a real, unpublished host spectrum could never reference itself in-library). **duplicate_aware**: cross-library PLUS excludes references flagged as near-duplicate technical replicates of the query (section 6 above) -- the most realistic protocol, and the one used for the primary host benchmark below.

In [9]:
near_dup_lookup = set()
if len(near_duplicate_audit):
    near_dup_lookup = set(
        near_duplicate_audit.loc[near_duplicate_audit["is_near_duplicate_reference"], ["query_id", "reference_spectrum_id"]]
        .itertuples(index=False, name=None)
    )
host_fold_by_query = host_holdout_spectra.set_index("query_id")["fold"].to_dict()

# PERFORMANCE: a handful of connectivity_keys have 1000+ reference spectra in train (verified:
# max 1,468) -- filtering the FULL unbounded list on every (query, candidate) row made the naive
# version of this cell take >40 minutes and still not finish. Every regime only ever needs
# MAX_REFS_PER_CANDIDATE=5 references per candidate, so pre-capping reference_index to a small,
# generous headroom (30 -- comfortably more than enough room for self/same-lib/near-dup
# exclusions to still leave >=5) makes every per-row filter O(30) worst case instead of O(1000+).
REFERENCE_INDEX_CAP = 30
reference_index_capped = {key: ids[:REFERENCE_INDEX_CAP] for key, ids in reference_index.items()}


def _capped_refs(candidate_key, query_row, exclude_same_lib, exclude_near_dup, limit=MAX_REFS_PER_CANDIDATE):
    out = []
    for rid in reference_index_capped.get(candidate_key, ()):
        if rid == query_row.query_id:
            continue
        if exclude_same_lib and spectrum_id_to_ingest_lib.get(rid) == HOST_INGEST_LIB:
            continue
        if exclude_near_dup and (query_row.query_id, rid) in near_dup_lookup:
            continue
        out.append(rid)
        if len(out) >= limit:
            break
    return out


def standard_ref_fn(candidate_key, query_row):
    return _capped_refs(candidate_key, query_row, exclude_same_lib=False, exclude_near_dup=False)


def cross_library_ref_fn(candidate_key, query_row):
    return _capped_refs(candidate_key, query_row, exclude_same_lib=True, exclude_near_dup=False)


def duplicate_aware_ref_fn(candidate_key, query_row):
    return _capped_refs(candidate_key, query_row, exclude_same_lib=True, exclude_near_dup=True)


def build_host_pair_features(regime_reference_fn, max_refs=MAX_REFS_PER_CANDIDATE):
    pool_by_query = {qid: g for qid, g in host_pool.groupby("query_id")}
    records = []
    for query_row in host_holdout_spectra.itertuples(index=False):
        q_offset = offset_of(query_row.query_id)
        if q_offset not in host_peaks_by_offset:
            continue
        q_peaks = host_peaks_by_offset[q_offset]
        my_candidates = pool_by_query.get(query_row.query_id)
        if my_candidates is None:
            continue
        for cand_row in my_candidates.itertuples(index=False):
            ref_ids = regime_reference_fn(cand_row.candidate_connectivity_key, query_row)[:max_refs]
            ref_peaks_list = [host_peaks_by_offset[offset_of(rid)] for rid in ref_ids if offset_of(rid) in host_peaks_by_offset]
            scores = compute_pair_spectral_scores(q_peaks, ref_peaks_list, bin_width=BIN_WIDTH_DA, peak_tol_da=PEAK_TOL_DA)
            record = {
                "query_id": cand_row.query_id, "candidate_connectivity_key": cand_row.candidate_connectivity_key,
                "abs_mass_error_ppm": cand_row.abs_mass_error_ppm,
                "is_true_candidate": getattr(cand_row, "is_true_candidate", False),
                "fold": host_fold_by_query[cand_row.query_id],
            }
            record.update(scores)
            records.append(record)
    return pd.DataFrame.from_records(records)


t0 = time.time()
host_pairs_standard = build_host_pair_features(standard_ref_fn)
host_pairs_cross_library = build_host_pair_features(cross_library_ref_fn)
host_pairs_duplicate_aware = build_host_pair_features(duplicate_aware_ref_fn)
print(f"built 3 protocol pair-feature tables in {time.time() - t0:.1f}s: "
      f"standard={len(host_pairs_standard):,}, cross_library={len(host_pairs_cross_library):,}, duplicate_aware={len(host_pairs_duplicate_aware):,} rows")
for name, df in [("host_pair_features_standard", host_pairs_standard), ("host_pair_features_cross_library", host_pairs_cross_library),
                  ("host_pair_features_duplicate_aware", host_pairs_duplicate_aware)]:
    save_artifact(df, name, HOST_INTERIM, description=f"Host holdout pair features, {name.split('_', 3)[-1]} protocol")
checks.append(CheckResult("3 reference protocols built", all(len(d) > 0 for d in [host_pairs_standard, host_pairs_cross_library, host_pairs_duplicate_aware]), ""))

built 3 protocol pair-feature tables in 541.2s: standard=126,573, cross_library=126,573, duplicate_aware=126,573 rows


## 8. Section 6 — score each protocol: mass-only, peak-overlap, LambdaMART

LambdaMART score uses the fold model that held OUT the host query's own connectivity's fold -- reproduced in section 4, never the query's own-fold model (section 4.2's exclusion, enforced structurally here, not just asserted).

In [10]:
def score_protocol(pair_df, protocol_name):
    lgbm_scores = pd.Series(np.nan, index=pair_df.index)
    for fold_value, model in fold_models.items():
        mask = pair_df["fold"] == fold_value
        if mask.any():
            lgbm_scores.loc[mask] = model.predict(pair_df.loc[mask, MODEL_FEATURES])
    assert lgbm_scores.notna().all(), "every host pair row must be scored by its OWN fold's held-out model"

    results = {}
    for method_name, score_col, ascending, series in [
        ("mass_only", None, True, -pair_df["abs_mass_error_ppm"]),
        ("peak_overlap_max", None, False, pair_df["peak_overlap_frac_max"]),
        ("lambdamart", None, False, lgbm_scores),
    ]:
        ranked = rank_candidates(pair_df.assign(score=series), score_col="score", ascending=False)
        metrics = ranking_metrics(ranked, all_host_query_ids, k_values=(1, 5, 10, 25))
        ranks = true_candidate_rank(ranked).reindex(all_host_query_ids)
        rr = per_query_reciprocal_rank(ranks, k=25)
        results[method_name] = {"metrics": metrics, "per_query_rr": rr}
    return results


protocol_results = {}
for protocol_name, pair_df in [("standard", host_pairs_standard), ("cross_library", host_pairs_cross_library), ("duplicate_aware", host_pairs_duplicate_aware)]:
    protocol_results[protocol_name] = score_protocol(pair_df, protocol_name)

host_summary_rows = []
for protocol_name, methods in protocol_results.items():
    for method_name, r in methods.items():
        e2e = r["metrics"]["end_to_end"]
        host_summary_rows.append({
            "protocol": protocol_name, "method": method_name, "mrr_at_25": e2e["mrr_at_25"],
            "hit_at_1": e2e["hit_at_1"], "hit_at_5": e2e["hit_at_5"], "hit_at_25": e2e["hit_at_25"],
            "candidate_recall": e2e["candidate_recall"], "n_queries": e2e["n_queries"],
        })
host_mode_a_metrics = pd.DataFrame(host_summary_rows)
display(host_mode_a_metrics)
checks.append(CheckResult("host Mode-A scored under all 3 protocols x 3 methods", len(host_mode_a_metrics) == 9, ""))

,protocol,method,mrr_at_25,hit_at_1,hit_at_5,hit_at_25,candidate_recall,n_queries
0,standard,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
1,standard,peak_overlap_max,0.592721,0.490709,0.720439,0.857264,1.0,1184
2,standard,lambdamart,0.708840,0.579392,0.874155,0.979730,1.0,1184
3,cross_library,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
4,cross_library,peak_overlap_max,0.586754,0.484797,0.714527,0.854730,1.0,1184
5,cross_library,lambdamart,0.706567,0.576858,0.871622,0.979730,1.0,1184
6,duplicate_aware,mass_only,0.399144,0.214527,0.647804,0.915541,1.0,1184
7,duplicate_aware,peak_overlap_max,0.581486,0.480574,0.708615,0.850507,1.0,1184
8,duplicate_aware,lambdamart,0.706548,0.579392,0.866554,0.978885,1.0,1184


## 9. Section 7 — paired bootstrap CIs (duplicate-aware protocol -- the primary host benchmark)

In [11]:
primary = protocol_results["duplicate_aware"]
bootstrap_rows = []
comparisons = [("lambdamart", "peak_overlap_max"), ("lambdamart", "mass_only"), ("peak_overlap_max", "mass_only")]
for a_name, b_name in comparisons:
    row = paired_bootstrap_report(
        "reciprocal_rank@25", a_name, primary[a_name]["per_query_rr"], b_name, primary[b_name]["per_query_rr"],
        n_bootstrap=N_BOOTSTRAP, seed=SEED,
    )
    row["protocol"] = "duplicate_aware"
    bootstrap_rows.append(row)
paired_bootstrap_comparisons = pd.DataFrame(bootstrap_rows)
display(paired_bootstrap_comparisons[["protocol", "method_a", "method_b", "mean_a", "mean_b", "mean_diff", "ci_low", "ci_high", "significant_at_95"]])

lgbm_vs_peak_overlap = paired_bootstrap_comparisons[
    (paired_bootstrap_comparisons["method_a"] == "lambdamart") & (paired_bootstrap_comparisons["method_b"] == "peak_overlap_max")
].iloc[0]
print(f"\nLambdaMART vs peak_overlap_max on REALISTIC host holdout: diff={lgbm_vs_peak_overlap['mean_diff']:+.4f} "
      f"[{lgbm_vs_peak_overlap['ci_low']:+.4f}, {lgbm_vs_peak_overlap['ci_high']:+.4f}] "
      f"-- {'SIGNIFICANT' if lgbm_vs_peak_overlap['significant_at_95'] else 'NOT significant'}")
checks.append(CheckResult("paired bootstrap CIs computed for host holdout", len(paired_bootstrap_comparisons) == 3, ""))

,protocol,method_a,method_b,mean_a,mean_b,mean_diff,ci_low,ci_high,significant_at_95
0,duplicate_aware,lambdamart,peak_overlap_max,0.706548,0.581486,0.125062,0.105434,0.144171,True
1,duplicate_aware,lambdamart,mass_only,0.706548,0.399144,0.307405,0.282057,0.331465,True
2,duplicate_aware,peak_overlap_max,mass_only,0.581486,0.399144,0.182342,0.154746,0.208224,True



LambdaMART vs peak_overlap_max on REALISTIC host holdout: diff=+0.1251 [+0.1054, +0.1442] -- SIGNIFICANT


## 10. Persist artifacts + register baseline

Optimism gap: how much of notebook 09's `0.7531` known-spectrum MRR@25 survives on a genuinely external, cross-instrument holdout the model never trained on?

In [12]:
save_artifact(host_mode_a_metrics, "host_mode_a_metrics", HOST_PROCESSED,
              description="Host holdout Mode-A MRR@25/Hit@K under 3 reference protocols x 3 scoring methods")
save_artifact(paired_bootstrap_comparisons, "paired_bootstrap_comparisons", HOST_PROCESSED,
              description="Paired bootstrap CIs for host holdout method comparisons (duplicate_aware protocol)")

dev_lgbm_mrr25 = mode_a_mrr25_registered
host_lgbm_mrr25 = host_mode_a_metrics.query("protocol == 'duplicate_aware' and method == 'lambdamart'")["mrr_at_25"].iloc[0]
optimism_gap = dev_lgbm_mrr25 - host_lgbm_mrr25
print(f"dev (in-distribution) LambdaMART MRR@25:  {dev_lgbm_mrr25:.4f}")
print(f"host (duplicate-aware) LambdaMART MRR@25: {host_lgbm_mrr25:.4f}")
print(f"optimism gap: {optimism_gap:+.4f}")

register_baseline("notebook_10_host_holdout_mode_a", {
    "mode_a_host_duplicate_aware": {
        "mrr_at_25": float(host_lgbm_mrr25), "status": "VALID_HOST_BENCHMARK", "use_for_model_selection": True,
        "protocol": "cross_library + near_duplicate_excluded", "n_queries": int(len(all_host_query_ids)),
        "n_connectivities": int(host_holdout_spectra['true_connectivity_key'].nunique()),
        "vs_dev_lambdamart_optimism_gap": float(optimism_gap),
    },
    "protocol_comparison": host_mode_a_metrics.to_dict("records"),
    "bootstrap_comparisons": paired_bootstrap_comparisons.drop(columns=["protocol"]).to_dict("records"),
}, BASELINE_REGISTRY_PATH, source_notebook="10_host_holdout_mode_a.ipynb", overwrite=True)
print(f"\nBaseline registered in {BASELINE_REGISTRY_PATH}")

with open(BASELINE_REGISTRY_PATH, encoding="utf-8") as f:
    full_registry = json.load(f)
with open(REPORTS_DIR / "baseline_registry.json", "w", encoding="utf-8") as f:
    json.dump(full_registry, f, indent=2, default=str)

checks.append(CheckResult("host holdout artifacts + baseline registered", True, f"optimism_gap={optimism_gap:+.4f}"))

dev (in-distribution) LambdaMART MRR@25:  0.7531
host (duplicate-aware) LambdaMART MRR@25: 0.7065
optimism gap: +0.0466

Baseline registered in C:\Users\myben\OneDrive\Documents\Enveda\data\processed\baseline_registry.json


## 11. Final validation checklist + summary

In [13]:
all_passed, checks_dict = summarize_checks(checks)
checklist_df = pd.DataFrame([{"Check": name, "Status": "PASS" if ok else "FAIL"} for name, ok in checks_dict.items()])
display(checklist_df)

TOTAL_RUNTIME_S = time.time() - NOTEBOOK_START_TIME
print("=" * 70)
print("HOST HOLDOUT MODE-A -- SUMMARY")
print("=" * 70)
print(f"\nHost holdout: {len(host_holdout_spectra):,} spectra / {host_holdout_spectra['true_connectivity_key'].nunique():,} connectivities ({HOST_INGEST_LIB}, timsTOF)")
print(f"Candidate availability @ {SELECTED_PPM}ppm: {host_candidate_availability:.2%}")
print(f"\nStandard vs. cross-library vs. duplicate-aware protocol comparison:")
print(host_mode_a_metrics.pivot(index="method", columns="protocol", values="mrr_at_25").to_string())
print(f"\nPrimary (duplicate-aware) host benchmark:")
print(f"    mass_only:        {protocol_results['duplicate_aware']['mass_only']['metrics']['end_to_end']['mrr_at_25']:.4f}")
print(f"    peak_overlap_max:  {protocol_results['duplicate_aware']['peak_overlap_max']['metrics']['end_to_end']['mrr_at_25']:.4f}")
print(f"    lambdamart:        {host_lgbm_mrr25:.4f}")
print(f"\nLambdaMART vs peak_overlap_max: diff={lgbm_vs_peak_overlap['mean_diff']:+.4f} "
      f"[{lgbm_vs_peak_overlap['ci_low']:+.4f}, {lgbm_vs_peak_overlap['ci_high']:+.4f}] "
      f"-- {'SIGNIFICANT' if lgbm_vs_peak_overlap['significant_at_95'] else 'not significant'}")
print(f"\nOptimism gap (dev MRR@25 {dev_lgbm_mrr25:.4f} -> host MRR@25 {host_lgbm_mrr25:.4f}): {optimism_gap:+.4f}")
print(f"\n{'ALL CHECKS PASS' if all_passed else 'SOME CHECKS FAILED -- see table above'}")
print(f"\nRuntime: {TOTAL_RUNTIME_S:.1f}s")
print(f"\nDeferred this milestone (spec Parts 8-19 -- NOT implemented here):")
print(f"    Part 8  test-like multiplicity resampling (multi-seed spectrum-count matching to the real test distribution)")
print(f"    Part 9  temperature calibration of per-spectrum candidate scores")
print(f"    Part 10 mass log-likelihood model (replacing raw -abs_ppm)")
print(f"    Part 11-12 molecule-level aggregation comparison (max/mean/top-k/product-of-evidence) -- max is still unreplaced")
print(f"    Part 13-14 molecule-level test inference + first offline submission")
print(f"    Part 15 offline Kaggle smoke test script")
print(f"    Part 16 runtime profiling artifact")
print("=" * 70)

,Check,Status
0,upstream artifacts loaded,PASS
1,host holdout artifacts saved,PASS
2,host/training connectivity overlap computed (i...,PASS
3,fold models reproduced,PASS
4,host candidate pool generated,PASS
5,host peak arrays loaded,PASS
6,near-duplicate reference audit computed,PASS
7,3 reference protocols built,PASS
8,host Mode-A scored under all 3 protocols x 3 m...,PASS
9,paired bootstrap CIs computed for host holdout,PASS


HOST HOLDOUT MODE-A -- SUMMARY

Host holdout: 1,184 spectra / 250 connectivities (enveda-np-examples, timsTOF)
Candidate availability @ 50ppm: 100.00%

Standard vs. cross-library vs. duplicate-aware protocol comparison:
protocol          cross_library  duplicate_aware  standard
method                                                    
lambdamart             0.706567         0.706548  0.708840
mass_only              0.399144         0.399144  0.399144
peak_overlap_max       0.586754         0.581486  0.592721

Primary (duplicate-aware) host benchmark:
    mass_only:        0.3991
    peak_overlap_max:  0.5815
    lambdamart:        0.7065

LambdaMART vs peak_overlap_max: diff=+0.1251 [+0.1054, +0.1442] -- SIGNIFICANT

Optimism gap (dev MRR@25 0.7531 -> host MRR@25 0.7065): +0.0466

ALL CHECKS PASS

Runtime: 630.5s

Deferred this milestone (spec Parts 8-19 -- NOT implemented here):
    Part 8  test-like multiplicity resampling (multi-seed spectrum-count matching to the real test distrib